# Inspección de `smoke_local_scraper.db`

Este notebook ayuda a explorar la base de datos generada por `tests/smoke_local_scraper.py`.

Incluye:
- conexión a SQLite
- listado de tablas y esquema
- filas de muestra
- análisis de columnas y nulos
- métricas básicas
- consultas de exploración
- exportación de resultados

## 1. Importar librerías y configurar la conexión

Importamos `sqlite3`, `pandas` y algunas utilidades para explorar la base desde el notebook.

In [10]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("smoke_local_scraper.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Conectado a: {DB_PATH.resolve()}")

Conectado a: /Users/Adri/Desktop/Github_repos/Immo_scraper/smoke_local_scraper.db


## 2. Conectarse a `smoke_local_scraper.db`

Abrimos la base SQLite del repositorio y dejamos una conexión reutilizable.

### Comprobación rápida de existencia

Si el archivo no existe, el notebook falla al instante con un mensaje claro.

In [11]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("smoke_local_scraper.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Conectado a: {DB_PATH.resolve()}")

Conectado a: /Users/Adri/Desktop/Github_repos/Immo_scraper/smoke_local_scraper.db


## 3. Listar tablas y esquema de la base de datos

Consultamos `sqlite_master` para ver las tablas disponibles y las sentencias `CREATE TABLE`.

In [12]:
tables_df = pd.read_sql_query(
    "SELECT name AS table_name, sql AS create_sql FROM sqlite_master WHERE type='table' ORDER BY name",
    conn,
)
display(tables_df)

for _, row in tables_df.iterrows():
    print(f"\n### {row['table_name']}")
    print(row['create_sql'])

,table_name,create_sql
0,price_history,CREATE TABLE price_history (\n id ...
1,properties,CREATE TABLE properties (\n property_id TE...
2,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"



### price_history
CREATE TABLE price_history (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    property_id TEXT NOT NULL REFERENCES properties(property_id),
    price       REAL NOT NULL,
    date        TEXT NOT NULL         -- ISO-8601 datetime
)

### properties
CREATE TABLE properties (
    property_id  TEXT PRIMARY KEY,
    source       TEXT NOT NULL,
    title        TEXT,
    url          TEXT,
    price        REAL,
    rooms        INTEGER,
    bathrooms    INTEGER,
    sqm          INTEGER,
    has_pool     INTEGER DEFAULT 0,   -- 0/1 (SQLite has no BOOLEAN)
    has_ac       INTEGER DEFAULT 0,
    orientation  TEXT,
    property_type TEXT,
    operation    TEXT,
    city         TEXT,
    district     TEXT,
    neighborhood TEXT,
    postal_code  TEXT,
    latitude     REAL,
    longitude    REAL,
    energy_rating TEXT,
    year_built   INTEGER,
    floor        TEXT,
    terrace      INTEGER DEFAULT 0,
    elevator     INTEGER DEFAULT 0,
    parking      INTEGER DEFA

## 4. Inspeccionar filas de muestra por tabla

Leemos unas pocas filas de cada tabla para ver el contenido real.

In [19]:
for table_name in tables_df['table_name']:
    print(f"\n### {table_name}")
    sample_df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 30", conn)
    display(sample_df)


### price_history


,id,property_id,price,date
0,1,amat_practica-i-accesible-planta-baixa-a-tocar...,310000.0,2026-04-29T19:06:06+00:00
1,2,amat_fantastica-casa-de-disseny-a-mirasol,1225000.0,2026-04-29T19:06:06+00:00
2,3,amat_casa-a-sant-cugat-dos-habitatges-en-un,845000.0,2026-04-29T19:06:06+00:00
3,4,amat_placa-daparcament-en-venda-al-centre,20000.0,2026-04-29T19:06:06+00:00
4,5,amat_magnifica-casa-al-golf-amb-grans-espais,1575000.0,2026-04-29T19:06:06+00:00
5,6,qgat_homes_ref-1717,1150000.0,2026-05-09T11:22:01+00:00
6,7,qgat_homes_ref-1682,1150000.0,2026-05-09T11:22:01+00:00
7,8,qgat_homes_ref-1716,615000.0,2026-05-09T11:22:01+00:00
8,9,qgat_homes_ref-1699,589000.0,2026-05-09T11:22:01+00:00
9,10,qgat_homes_ref-1685,775000.0,2026-05-09T11:22:01+00:00



### properties


,property_id,source,title,url,price,rooms,bathrooms,sqm,has_pool,has_ac,...,floor,terrace,elevator,parking,is_favourite,similarity_score,similarity_profile,first_seen,last_seen,status
0,amat_practica-i-accesible-planta-baixa-a-tocar...,amat,"Pràctica i accesible planta baixa, a tocar del...",https://www.amatimmobiliaris.com/ca/venda/apar...,310000.0,2.0,1.0,59.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
1,amat_fantastica-casa-de-disseny-a-mirasol,amat,Fantàstica casa de disseny a Mirasol,https://www.amatimmobiliaris.com/ca/venda/casa...,1225000.0,5.0,4.0,254.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
2,amat_casa-a-sant-cugat-dos-habitatges-en-un,amat,Casa a Sant Cugat: Dos habitatges en un,https://www.amatimmobiliaris.com/ca/venda/casa...,845000.0,6.0,3.0,285.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
3,amat_placa-daparcament-en-venda-al-centre,amat,Plaça d'aparcament en venda al centre,https://www.amatimmobiliaris.com/ca/venda/plac...,20000.0,NaN,NaN,NaN,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
4,amat_magnifica-casa-al-golf-amb-grans-espais,amat,"Magnífica casa al Golf, amb grans espais",https://www.amatimmobiliaris.com/ca/venda/casa...,1575000.0,6.0,6.0,566.0,0,0,...,None,0,0,0,0,None,None,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
5,qgat_homes_ref-1717,qgat_homes,Casa amb vistes a Mirasol,https://www.qgathomes.com/ca/venda/a-barcelona...,1150000.0,5.0,4.0,278.0,0,0,...,None,0,0,0,0,None,None,2026-05-09T11:22:01+00:00,2026-05-09T11:30:29+00:00,active
6,qgat_homes_ref-1682,qgat_homes,Casa amb vistes a Mirasol,https://www.qgathomes.com/ca/venda/a-barcelona...,1150000.0,5.0,4.0,278.0,0,0,...,None,0,0,0,0,None,None,2026-05-09T11:22:01+00:00,2026-05-09T11:30:29+00:00,active
7,qgat_homes_ref-1716,qgat_homes,Àtic en venda a Sant Cugat del Vallès - Can Mates,https://www.qgathomes.com/ca/venda/a-barcelona...,615000.0,3.0,2.0,111.0,0,0,...,None,0,0,0,0,None,None,2026-05-09T11:22:01+00:00,2026-05-09T11:30:29+00:00,active
8,qgat_homes_ref-1699,qgat_homes,Cases en venda a Sant Cugat del Vallès - Valld...,https://www.qgathomes.com/ca/venda/a-barcelona...,589000.0,1.0,1.0,60.0,0,0,...,None,0,0,0,0,None,None,2026-05-09T11:22:01+00:00,2026-05-09T11:30:29+00:00,active
9,qgat_homes_ref-1685,qgat_homes,Casa adossada a Sant Domenech,https://www.qgathomes.com/ca/venda/a-barcelona...,775000.0,4.0,2.0,171.0,0,0,...,None,0,0,0,0,None,None,2026-05-09T11:22:01+00:00,2026-05-09T11:30:29+00:00,active



### sqlite_sequence


,name,seq
0,price_history,17


## 5. Analizar columnas, tipos y valores nulos

Creamos un resumen por tabla con tipos y proporción de nulos.

In [14]:
def inspect_table(table_name: str) -> pd.DataFrame:
    columns_df = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)
    row_count = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table_name}", conn).iloc[0]["row_count"]
    if row_count == 0:
        columns_df["null_count"] = 0
        columns_df["null_pct"] = 0.0
        return columns_df

    null_exprs = ", ".join(
        [f"SUM(CASE WHEN {col} IS NULL OR TRIM(CAST({col} AS TEXT)) = '' THEN 1 ELSE 0 END) AS {col}" for col in columns_df['name']]
    )
    null_counts = pd.read_sql_query(f"SELECT {null_exprs} FROM {table_name}", conn).iloc[0].to_dict()
    columns_df["null_count"] = columns_df["name"].map(null_counts).fillna(0).astype(int)
    columns_df["null_pct"] = (columns_df["null_count"] / row_count).round(3)
    return columns_df

for table_name in tables_df['table_name']:
    print(f"\n### {table_name}")
    display(inspect_table(table_name))


### price_history


,cid,name,type,notnull,dflt_value,pk,null_count,null_pct
0,0,id,INTEGER,0,None,1,0,0.0
1,1,property_id,TEXT,1,None,0,0,0.0
2,2,price,REAL,1,None,0,0,0.0
3,3,date,TEXT,1,None,0,0,0.0



### properties


,cid,name,type,notnull,dflt_value,pk,null_count,null_pct
0,0,property_id,TEXT,0,None,1,0,0.000
1,1,source,TEXT,1,None,0,0,0.000
2,2,title,TEXT,0,None,0,0,0.000
3,3,url,TEXT,0,None,0,0,0.000
4,4,price,REAL,0,None,0,0,0.000
5,5,rooms,INTEGER,0,None,0,4,0.235
6,6,bathrooms,INTEGER,0,None,0,3,0.176
7,7,sqm,INTEGER,0,None,0,1,0.059
8,8,has_pool,INTEGER,0,0,0,0,0.000
9,9,has_ac,INTEGER,0,0,0,0,0.000



### sqlite_sequence


,cid,name,type,notnull,dflt_value,pk,null_count,null_pct
0,0,name,,0,None,0,0,0.0
1,1,seq,,0,None,0,0,0.0


## 6. Calcular conteos y métricas básicas

Calculamos tamaño, unicidad y algunas estadísticas rápidas para la tabla principal.

In [15]:
properties_df = pd.read_sql_query("SELECT * FROM properties", conn)
summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_property_id",
            "active_rows",
            "inactive_rows",
            "price_min",
            "price_median",
            "price_max",
            "rooms_filled",
            "bathrooms_filled",
            "sqm_filled",
        ],
        "value": [
            len(properties_df),
            properties_df.shape[1],
            properties_df["property_id"].nunique(dropna=True),
            (properties_df["status"] == "active").sum() if "status" in properties_df else None,
            (properties_df["status"] == "inactive").sum() if "status" in properties_df else None,
            properties_df["price"].min(),
            properties_df["price"].median(),
            properties_df["price"].max(),
            properties_df["rooms"].notna().sum(),
            properties_df["bathrooms"].notna().sum(),
            properties_df["sqm"].notna().sum(),
        ],
    }
)
display(summary)
display(properties_df.describe(include="all"))

,metric,value
0,rows,17.0
1,columns,31.0
2,unique_property_id,17.0
3,active_rows,17.0
4,inactive_rows,0.0
5,price_min,8000.0
6,price_median,750000.0
7,price_max,1575000.0
8,rooms_filled,13.0
9,bathrooms_filled,14.0


,property_id,source,title,url,price,rooms,bathrooms,sqm,has_pool,has_ac,...,floor,terrace,elevator,parking,is_favourite,similarity_score,similarity_profile,first_seen,last_seen,status
count,17,17,17,17,1.700000e+01,13.000000,14.000000,16.000000,17.000000,17.0,...,0,17.0,17.0,17.0,17.0,0,0,17,17,17
unique,17,2,15,17,NaN,NaN,NaN,NaN,NaN,NaN,...,0,NaN,NaN,NaN,NaN,0,0,2,2,1
top,amat_practica-i-accesible-planta-baixa-a-tocar...,qgat_homes,Casa amb vistes a Mirasol,https://www.amatimmobiliaris.com/ca/venda/apar...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-09T11:22:01+00:00,2026-05-09T11:30:29+00:00,active
freq,1,12,2,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,12,17
mean,NaN,NaN,NaN,NaN,7.511765e+05,4.000000,2.714286,202.000000,0.058824,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,5.007007e+05,1.527525,1.382783,149.396564,0.242536,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,8.000000e+03,1.000000,1.000000,10.000000,0.000000,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,3.950000e+05,3.000000,2.000000,96.000000,0.000000,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,7.500000e+05,4.000000,2.000000,197.500000,0.000000,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,1.150000e+06,5.000000,3.750000,279.750000,0.000000,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## 7. Ejecutar consultas de exploración y filtrado

Consultas SQL útiles para ver datos activos, detectar duplicados o encontrar registros sospechosos.

In [16]:
available_tables = set(tables_df['table_name'])

queries = {
    'active_properties': """
        SELECT property_id, source, title, price, rooms, bathrooms, sqm, city, status
        FROM properties
        WHERE status = 'active'
        ORDER BY title
    """,
    'missing_required_fields': """
        SELECT property_id, title, price, rooms, bathrooms, sqm
        FROM properties
        WHERE title IS NULL OR price IS NULL OR rooms IS NULL OR bathrooms IS NULL OR sqm IS NULL
        ORDER BY title
    """,
}

if 'price_history' in available_tables:
    price_history_columns = pd.read_sql_query("PRAGMA table_info(price_history)", conn)['name'].tolist()
    timestamp_candidates = [
        'seen_at', 'scraped_at', 'created_at', 'updated_at', 'timestamp', 'date', 'captured_at'
    ]
    order_col = next((col for col in timestamp_candidates if col in price_history_columns), None)

    selected_cols = ['property_id', 'price']
    if order_col:
        selected_cols.append(order_col)

    order_clause = f"ORDER BY {order_col} DESC" if order_col else ""
    queries['price_history'] = f"""
        SELECT {', '.join(selected_cols)}
        FROM price_history
        {order_clause}
    """

for name, sql in queries.items():
    print(f"\n### {name}")
    try:
        display(pd.read_sql_query(sql, conn))
    except Exception as exc:
        print(f"Error ejecutando {name}: {exc}")


### active_properties


,property_id,source,title,price,rooms,bathrooms,sqm,city,status
0,qgat_homes_ref-1668,qgat_homes,"Adossada amb piscina a Eixample, Sant Cugat de...",1285000.0,4.0,2.0,224.0,Sant Cugat del Vallès,active
1,amat_casa-a-sant-cugat-dos-habitatges-en-un,amat,Casa a Sant Cugat: Dos habitatges en un,845000.0,6.0,3.0,285.0,Sant Cugat del Vallès,active
2,qgat_homes_ref-1685,qgat_homes,Casa adossada a Sant Domenech,775000.0,4.0,2.0,171.0,Sant Cugat del Vallès,active
3,qgat_homes_ref-1717,qgat_homes,Casa amb vistes a Mirasol,1150000.0,5.0,4.0,278.0,Sant Cugat del Vallès,active
4,qgat_homes_ref-1682,qgat_homes,Casa amb vistes a Mirasol,1150000.0,5.0,4.0,278.0,Sant Cugat del Vallès,active
5,qgat_homes_ref-1699,qgat_homes,Cases en venda a Sant Cugat del Vallès - Valld...,589000.0,1.0,1.0,60.0,Sant Cugat del Vallès,active
6,amat_fantastica-casa-de-disseny-a-mirasol,amat,Fantàstica casa de disseny a Mirasol,1225000.0,5.0,4.0,254.0,Sant Cugat del Vallès,active
7,qgat_homes_ref-1710,qgat_homes,Garatge en venda i lloguer a Sant Cugat del Va...,8000.0,NaN,NaN,10.0,Sant Cugat del Vallès,active
8,qgat_homes_ref-1709,qgat_homes,Garatge en venda i lloguer a Sant Cugat del Va...,14000.0,NaN,NaN,11.0,Sant Cugat del Vallès,active
9,qgat_homes_ref-1714,qgat_homes,Local Comercial en venda a Sant Cugat del Vall...,395000.0,NaN,2.0,334.0,Sant Cugat del Vallès,active



### missing_required_fields


,property_id,title,price,rooms,bathrooms,sqm
0,qgat_homes_ref-1710,Garatge en venda i lloguer a Sant Cugat del Va...,8000.0,None,NaN,10.0
1,qgat_homes_ref-1709,Garatge en venda i lloguer a Sant Cugat del Va...,14000.0,None,NaN,11.0
2,qgat_homes_ref-1714,Local Comercial en venda a Sant Cugat del Vall...,395000.0,None,2.0,334.0
3,amat_placa-daparcament-en-venda-al-centre,Plaça d'aparcament en venda al centre,20000.0,None,NaN,NaN



### price_history


,property_id,price,date
0,qgat_homes_ref-1717,1150000.0,2026-05-09T11:22:01+00:00
1,qgat_homes_ref-1682,1150000.0,2026-05-09T11:22:01+00:00
2,qgat_homes_ref-1716,615000.0,2026-05-09T11:22:01+00:00
3,qgat_homes_ref-1699,589000.0,2026-05-09T11:22:01+00:00
4,qgat_homes_ref-1685,775000.0,2026-05-09T11:22:01+00:00
5,qgat_homes_ref-1714,395000.0,2026-05-09T11:22:01+00:00
6,qgat_homes_ref-1710,8000.0,2026-05-09T11:22:01+00:00
7,qgat_homes_ref-1709,14000.0,2026-05-09T11:22:01+00:00
8,qgat_homes_ref-1635,1445000.0,2026-05-09T11:22:01+00:00
9,qgat_homes_ref-1707,619000.0,2026-05-09T11:22:01+00:00


## 8. Exportar resultados de inspección

Guardamos CSVs con resúmenes y tablas de salida para analizarlos fuera del notebook.

In [17]:
export_dir = Path('notebook_exports')
export_dir.mkdir(exist_ok=True)

tables_df.to_csv(export_dir / 'tables.csv', index=False)
summary.to_csv(export_dir / 'summary.csv', index=False)
properties_df.to_csv(export_dir / 'properties.csv', index=False)

print(f'Exported CSV files to: {export_dir.resolve()}')

Exported CSV files to: /Users/Adri/Desktop/Github_repos/Immo_scraper/notebook_exports


In [18]:
# Verificar columna de alta inicial y calcular antigüedad de anuncios
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
first_seen_candidates = ['first_seen', 'first_seen_at', 'created_at', 'published_at', 'date']
first_seen_col = next((col for col in first_seen_candidates if col in properties_columns), None)

if not first_seen_col:
    print("No se encontró una columna de primera publicación en properties.")
else:
    print(f"Usando columna de primera publicación: {first_seen_col}")
    sql_first_seen = f"""
        SELECT
            property_id,
            title,
            {first_seen_col} AS first_seen,
            CAST(julianday('now') - julianday({first_seen_col}) AS INTEGER) AS days_online
        FROM properties
        WHERE {first_seen_col} IS NOT NULL
        ORDER BY days_online DESC, first_seen ASC
        LIMIT 30
    """
    display(pd.read_sql_query(sql_first_seen, conn))

Usando columna de primera publicación: first_seen


,property_id,title,first_seen,days_online
0,amat_practica-i-accesible-planta-baixa-a-tocar...,"Pràctica i accesible planta baixa, a tocar del...",2026-04-29T19:06:06+00:00,9
1,amat_fantastica-casa-de-disseny-a-mirasol,Fantàstica casa de disseny a Mirasol,2026-04-29T19:06:06+00:00,9
2,amat_casa-a-sant-cugat-dos-habitatges-en-un,Casa a Sant Cugat: Dos habitatges en un,2026-04-29T19:06:06+00:00,9
3,amat_placa-daparcament-en-venda-al-centre,Plaça d'aparcament en venda al centre,2026-04-29T19:06:06+00:00,9
4,amat_magnifica-casa-al-golf-amb-grans-espais,"Magnífica casa al Golf, amb grans espais",2026-04-29T19:06:06+00:00,9
5,qgat_homes_ref-1717,Casa amb vistes a Mirasol,2026-05-09T11:22:01+00:00,0
6,qgat_homes_ref-1682,Casa amb vistes a Mirasol,2026-05-09T11:22:01+00:00,0
7,qgat_homes_ref-1716,Àtic en venda a Sant Cugat del Vallès - Can Mates,2026-05-09T11:22:01+00:00,0
8,qgat_homes_ref-1699,Cases en venda a Sant Cugat del Vallès - Valld...,2026-05-09T11:22:01+00:00,0
9,qgat_homes_ref-1685,Casa adossada a Sant Domenech,2026-05-09T11:22:01+00:00,0


## 9. Anuncios activos más antiguos

Esta vista muestra los anuncios que siguen activos y llevan más tiempo publicados, combinando `first_seen` y `last_seen`.

In [23]:
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
active_column = 'status' if 'status' in properties_columns else None

first_seen_candidates = ['first_seen', 'first_seen_at', 'created_at', 'published_at', 'date']
last_seen_candidates = ['last_seen', 'last_seen_at', 'updated_at', 'scraped_at', 'date']

first_seen_col = next((col for col in first_seen_candidates if col in properties_columns), None)
last_seen_col = next((col for col in last_seen_candidates if col in properties_columns), None)

if not first_seen_col and not last_seen_col:
    print('No se encontraron columnas útiles de seguimiento temporal en properties.')
else:
    print(f'Usando first_seen={first_seen_col or "n/a"} y last_seen={last_seen_col or "n/a"}')
    select_cols = [
        'property_id',
        'title',
        'source',
        'city',
        'price',
        'rooms',
        'bathrooms',
        'sqm',
    ]
    if active_column:
        select_cols.append('status')
    if first_seen_col:
        select_cols.append(f'{first_seen_col} AS first_seen')
        select_cols.append("CAST(julianday('now') - julianday(first_seen) AS INTEGER) AS days_online")
    else:
        select_cols.append('NULL AS first_seen')
        select_cols.append('NULL AS days_online')
    if last_seen_col:
        if last_seen_col == first_seen_col:
            select_cols.append(f'{last_seen_col} AS last_seen')
        else:
            select_cols.append(f'{last_seen_col} AS last_seen')
        select_cols.append("CAST(julianday('now') - julianday(last_seen) AS INTEGER) AS days_since_last_seen")
    else:
        select_cols.append('NULL AS last_seen')
        select_cols.append('NULL AS days_since_last_seen')

    where_clauses = []
    if active_column:
        where_clauses.append("status = 'active'")
    if first_seen_col:
        where_clauses.append(f'{first_seen_col} IS NOT NULL')
    if last_seen_col:
        where_clauses.append(f'{last_seen_col} IS NOT NULL')

    where_sql = ''
    if where_clauses:
        where_sql = 'WHERE ' + ' AND '.join(where_clauses)

    sql_oldest_active = f"""
        SELECT {', '.join(select_cols)}
        FROM properties
        {where_sql}
        ORDER BY days_online DESC, days_since_last_seen DESC, first_seen ASC, title ASC
        LIMIT 25
    """

    oldest_active_df = pd.read_sql_query(sql_oldest_active, conn)
    summary_columns = [
        col for col in [
            'property_id',
            'title',
            'first_seen',
            'days_online',
            'price',
        ]
        if col in oldest_active_df.columns
    ]
    display(oldest_active_df[summary_columns].head(10))

Usando first_seen=first_seen y last_seen=last_seen


,property_id,title,first_seen,days_online,price
0,amat_casa-a-sant-cugat-dos-habitatges-en-un,Casa a Sant Cugat: Dos habitatges en un,2026-04-29T19:06:06+00:00,9,845000.0
1,amat_fantastica-casa-de-disseny-a-mirasol,Fantàstica casa de disseny a Mirasol,2026-04-29T19:06:06+00:00,9,1225000.0
2,amat_magnifica-casa-al-golf-amb-grans-espais,"Magnífica casa al Golf, amb grans espais",2026-04-29T19:06:06+00:00,9,1575000.0
3,amat_placa-daparcament-en-venda-al-centre,Plaça d'aparcament en venda al centre,2026-04-29T19:06:06+00:00,9,20000.0
4,amat_practica-i-accesible-planta-baixa-a-tocar...,"Pràctica i accesible planta baixa, a tocar del...",2026-04-29T19:06:06+00:00,9,310000.0
5,qgat_homes_ref-1668,"Adossada amb piscina a Eixample, Sant Cugat de...",2026-05-09T11:22:01+00:00,0,1285000.0
6,qgat_homes_ref-1685,Casa adossada a Sant Domenech,2026-05-09T11:22:01+00:00,0,775000.0
7,qgat_homes_ref-1717,Casa amb vistes a Mirasol,2026-05-09T11:22:01+00:00,0,1150000.0
8,qgat_homes_ref-1682,Casa amb vistes a Mirasol,2026-05-09T11:22:01+00:00,0,1150000.0
9,qgat_homes_ref-1699,Cases en venda a Sant Cugat del Vallès - Valld...,2026-05-09T11:22:01+00:00,0,589000.0


## 10. Precio inicial vs precio actual

Esta vista compara el precio de la primera vez que se vio el anuncio con el precio actual guardado en la tabla.

In [25]:
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
price_first_seen_col = 'price_first_seen' if 'price_first_seen' in properties_columns else None

if not price_first_seen_col:
    print('No se encontró la columna price_first_seen en properties.')
else:
    sql_price_evolution = """
        SELECT
            property_id,
            title,
            source,
            city,
            price_first_seen,
            price,
            CASE
                WHEN price_first_seen IS NOT NULL AND price IS NOT NULL
                THEN price - price_first_seen
                ELSE NULL
            END AS price_delta,
            CASE
                WHEN price_first_seen IS NOT NULL AND price IS NOT NULL AND price_first_seen != 0
                THEN ROUND((price - price_first_seen) * 100.0 / price_first_seen, 2)
                ELSE NULL
            END AS price_delta_pct
        FROM properties
        WHERE status = 'active'
          AND price_first_seen IS NOT NULL
        ORDER BY ABS(COALESCE(price - price_first_seen, 0)) DESC, title ASC
        LIMIT 25
    """
    price_evolution_df = pd.read_sql_query(sql_price_evolution, conn)
    display(price_evolution_df[[
        'property_id',
        'title',
        'price_first_seen',
        'price',
        'price_delta',
        'price_delta_pct',
    ]].head(10))

,property_id,title,price_first_seen,price,price_delta,price_delta_pct
0,finques_soler_52,116 m²,720000.0,720000.0,0.0,0.0
1,finques_soler_82,124 m² y 4 habs,780000.0,780000.0,0.0,0.0
2,finques_soler_59,63 m² y 2 habs,269000.0,269000.0,0.0,0.0
3,finques_soler_58,74 m² y 2 habs,259000.0,259000.0,0.0,0.0
4,qgat_homes_ref-1668,"Adossada amb piscina a Eixample, Sant Cugat de...",1285000.0,1285000.0,0.0,0.0
5,qgat_homes_ref-1610,Adossat en venda a Sant Cugat del Vallès - Par...,930000.0,930000.0,0.0,0.0
6,finques_soler_76,Apartamento y 75 m²,279000.0,279000.0,0.0,0.0
7,amat_casa-a-sant-cugat-dos-habitatges-en-un,Casa a Sant Cugat: Dos habitatges en un,845000.0,845000.0,0.0,0.0
8,qgat_homes_ref-1685,Casa adossada a Sant Domenech,775000.0,775000.0,0.0,0.0
9,amat_casa-amb-encant-a-zona-mas-gener-de-mirasol,"Casa amb encant, a zona Mas Gener de Mirasol",665000.0,665000.0,0.0,0.0
